In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/Screw_Head_Detection/dataset"

print(os.listdir(dataset_path))

In [ ]:
import os

train_images = os.listdir(dataset_path + "/train/images")
valid_images = os.listdir(dataset_path + "/valid/images")

print("Train:", len(train_images))
print("Valid:", len(valid_images))

In [ ]:
import yaml

with open(dataset_path + "/data.yaml") as f:
    data = yaml.safe_load(f)

print(data)

In [ ]:
import os

train_labels = len(os.listdir(dataset_path + "/train/labels"))
valid_labels = len(os.listdir(dataset_path + "/valid/labels"))

print("Train labels:", train_labels)
print("Valid labels:", valid_labels)

In [ ]:
import os

combined_path = "/content/drive/MyDrive/Screw_Head_Detection/combined_dataset"

os.makedirs(combined_path, exist_ok=True)
os.makedirs(combined_path + "/images", exist_ok=True)
os.makedirs(combined_path + "/labels", exist_ok=True)

print("Combined dataset folder created.")

In [ ]:
import shutil
import glob

train_imgs = glob.glob(dataset_path + "/train/images/*")
valid_imgs = glob.glob(dataset_path + "/valid/images/*")

for img in train_imgs:
    shutil.copy(img, combined_path + "/images")

for img in valid_imgs:
    shutil.copy(img, combined_path + "/images")

print("Images copied.")

In [ ]:
train_labels = glob.glob(dataset_path + "/train/labels/*")
valid_labels = glob.glob(dataset_path + "/valid/labels/*")

for lbl in train_labels:
    shutil.copy(lbl, combined_path + "/labels")

for lbl in valid_labels:
    shutil.copy(lbl, combined_path + "/labels")

print("Labels copied.")

In [ ]:
print("Images:", len(os.listdir(combined_path + "/images")))
print("Labels:", len(os.listdir(combined_path + "/labels")))

In [ ]:
import os
import shutil
from sklearn.model_selection import KFold

In [ ]:
image_dir = combined_path + "/images"
label_dir = combined_path + "/labels"

images = sorted(os.listdir(image_dir))

print("Total Images:", len(images))

In [ ]:
folds_path = "/content/drive/MyDrive/Screw_Head_Detection/folds"

os.makedirs(folds_path, exist_ok=True)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
for fold, (_, test_idx) in enumerate(kf.split(images), start=1):

    fold_path = os.path.join(folds_path, f"fold{fold}")

    img_folder = os.path.join(fold_path, "images")
    lbl_folder = os.path.join(fold_path, "labels")

    os.makedirs(img_folder, exist_ok=True)
    os.makedirs(lbl_folder, exist_ok=True)

    for idx in test_idx:

        image_name = images[idx]

        shutil.copy(
            os.path.join(image_dir, image_name),
            os.path.join(img_folder, image_name)
        )

        label_name = os.path.splitext(image_name)[0] + ".txt"

        shutil.copy(
            os.path.join(label_dir, label_name),
            os.path.join(lbl_folder, label_name)
        )

print("All 5 folds created successfully!")

In [ ]:
for i in range(1, 6):

    fold = os.path.join(folds_path, f"fold{i}")

    num_images = len(os.listdir(os.path.join(fold, "images")))
    num_labels = len(os.listdir(os.path.join(fold, "labels")))

    print(f"Fold {i}: Images = {num_images}, Labels = {num_labels}")

In [ ]:
import os
import shutil

cv_path = "/content/drive/MyDrive/Screw_Head_Detection/cv_runs"

os.makedirs(cv_path, exist_ok=True)

In [ ]:
import yaml

for val_fold in range(1, 6):

    run_path = os.path.join(cv_path, f"run{val_fold}")

    train_img = os.path.join(run_path, "train/images")
    train_lbl = os.path.join(run_path, "train/labels")

    val_img = os.path.join(run_path, "val/images")
    val_lbl = os.path.join(run_path, "val/labels")

    os.makedirs(train_img, exist_ok=True)
    os.makedirs(train_lbl, exist_ok=True)

    os.makedirs(val_img, exist_ok=True)
    os.makedirs(val_lbl, exist_ok=True)

    for fold in range(1, 6):

        img_folder = os.path.join(folds_path, f"fold{fold}", "images")
        lbl_folder = os.path.join(folds_path, f"fold{fold}", "labels")

        destination_img = val_img if fold == val_fold else train_img
        destination_lbl = val_lbl if fold == val_fold else train_lbl

        for file in os.listdir(img_folder):
            shutil.copy(
                os.path.join(img_folder, file),
                os.path.join(destination_img, file)
            )

        for file in os.listdir(lbl_folder):
            shutil.copy(
                os.path.join(lbl_folder, file),
                os.path.join(destination_lbl, file)
            )

    data = {
        "path": run_path,
        "train": "train/images",
        "val": "val/images",
        "nc": 1,
        "names": ["screw"]
    }

    with open(os.path.join(run_path, "data.yaml"), "w") as f:
        yaml.dump(data, f)

print("All 5 run datasets created.")

In [ ]:
for i in range(1,6):

    run=f"/content/drive/MyDrive/Screw_Head_Detection/cv_runs/run{i}"

    train=len(os.listdir(run+"/train/images"))
    val=len(os.listdir(run+"/val/images"))

    print(f"Run {i}")
    print("Train:",train)
    print("Val:",val)
    print()

In [ ]:
from ultralytics import YOLO

for fold in range(1,6):

    print("="*60)
    print(f"Training Fold {fold}")
    print("="*60)

    model = YOLO("yolov8n.pt")

    model.train(
        data=f"/content/drive/MyDrive/Screw_Head_Detection/cv_runs/run{fold}/data.yaml",

        epochs=50,

        imgsz=640,

        batch=16,

        project="/content/drive/MyDrive/Screw_Head_Detection/results",

        name=f"fold_{fold}",

        exist_ok=True
    )

In [ ]:
import os

results_path = "/content/drive/MyDrive/Screw_Head_Detection/results"

for folder in sorted(os.listdir(results_path)):
    print(folder)

In [ ]:
import pandas as pd
import os

results_path = "/content/drive/MyDrive/Screw_Head_Detection/results"

summary = []

for i in range(1, 6):

    csv_path = os.path.join(results_path, f"fold_{i}", "results.csv")

    df = pd.read_csv(csv_path)

    last = df.iloc[-1]

    summary.append({
        "Fold": i,
        "Precision": last["metrics/precision(B)"],
        "Recall": last["metrics/recall(B)"],
        "mAP50": last["metrics/mAP50(B)"],
        "mAP50-95": last["metrics/mAP50-95(B)"],
    })

summary = pd.DataFrame(summary)

summary

In [ ]:
avg = summary.mean(numeric_only=True)
std = summary.std(numeric_only=True)

print("Average")
print(avg)

print("\nStandard Deviation")
print(std)

In [ ]:
import numpy as np

# Remove old summary rows if they already exist
summary = summary.drop(index=["Average", "Std Dev"], errors="ignore")

# Calculate average and standard deviation using only the 5 folds
avg_row = summary.drop(columns=["Fold"]).mean()
std_row = summary.drop(columns=["Fold"]).std()

# Add the rows back
summary.loc["Average"] = [np.nan, *avg_row.values]
summary.loc["Std Dev"] = [np.nan, *std_row.values]

# Replace NaN in Fold column with "-"
summary.loc["Average", "Fold"] = "-"
summary.loc["Std Dev", "Fold"] = "-"

# Save the final table
summary.to_csv(
    "/content/drive/MyDrive/Screw_Head_Detection/cross_validation_results.csv",
    index=False
)

summary